# Lecture 11 — Hamiltonian Simulation via Product Formulas (UnitaryLab companion)

Given a Hamiltonian $H$, **Hamiltonian simulation** builds a circuit that
approximates the time-evolution operator $U(t)=e^{-\mathrm{i}Ht}$. When
$H = A + B$ with $[A,B]\neq 0$, we cannot simply exponentiate the parts
separately. **Product (Trotter–Suzuki) formulas** split the evolution into a
sequence of easy exponentials:

- first order (Lie): $e^{-\mathrm{i}Ht} \approx \big(e^{-\mathrm{i}A t/r}\,e^{-\mathrm{i}B t/r}\big)^{r}$, error $\mathcal{O}(t^2/r)$;
- second order (Strang): $\big(e^{-\mathrm{i}A t/2r}\,e^{-\mathrm{i}B t/r}\,e^{-\mathrm{i}A t/2r}\big)^{r}$, error $\mathcal{O}(t^3/r^2)$.

We test this on the **transverse-field Ising model (TFIM)** on 2 qubits,
$H = -J\,Z\!\otimes\!Z - h\,(X\!\otimes\!I + I\!\otimes\!X)$, whose two pieces do
not commute. We use `ua.TrotterAlgorithm` (orders 1 and 2) and `ua.QDriftAlgorithm`,
comparing the returned approximate propagator against the exact
`scipy.linalg.expm(-1j*H*t)` (with a numpy `eigh` fallback).

Contents:
1. Setup and the exact reference
2. Introspect the Trotter result dictionary
3. Error vs. number of steps (order 1 vs. order 2)
4. Randomized compilation: qDrift
5. Your turn

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive")
import contextlib, io
import unitarylab_algorithms as ua

# The *Algorithm classes log verbosely and write artifacts under results/;
# silence stdout/stderr so notebook cells stay clean.
@contextlib.contextmanager
def quiet():
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        yield

# Exact matrix exponential: prefer scipy, fall back to a Hermitian eigh expm.
try:
    from scipy.linalg import expm as _expm
    def expm_herm(M):
        return _expm(M)
    _EXPM = "scipy.linalg.expm"
except Exception:
    def expm_herm(M):
        # M = -1j*H*t with H Hermitian; diagonalize H = V diag(w) V^dag.
        H = 1j * M            # recover H*t (Hermitian)
        w, V = np.linalg.eigh(H)
        return V @ np.diag(np.exp(-1j * w)) @ V.conj().T
    _EXPM = "numpy eigh fallback"

np.set_printoptions(precision=4, suppress=True)
print("exact-expm backend:", _EXPM)

exact-expm backend: scipy.linalg.expm


In [2]:
# TFIM on 2 qubits: H = -J Z⊗Z - h (X⊗I + I⊗X)
Z = np.diag([1, -1]).astype(complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
I2 = np.eye(2, dtype=complex)

def kron(*ops):
    r = ops[0]
    for o in ops[1:]:
        r = np.kron(r, o)
    return r

J, h = 1.0, 0.6
H = -J * kron(Z, Z) - h * (kron(X, I2) + kron(I2, X))
t = 1.0

# Non-commuting split A = -J ZZ (diagonal), B = -h (XI + IX).
A = -J * kron(Z, Z)
B = -h * (kron(X, I2) + kron(I2, X))
print("H Hermitian:", np.allclose(H, H.conj().T))
print("[A,B] != 0 :", not np.allclose(A @ B, B @ A))

U_exact = expm_herm(-1j * H * t)
print("U_exact unitary:", np.allclose(U_exact @ U_exact.conj().T, np.eye(4)))

H Hermitian: True
[A,B] != 0 : True
U_exact unitary: True


## 2. Introspect the Trotter result dictionary

`ua.TrotterAlgorithm().run(H, t, error, order, steps)` returns the approximate and
exact propagators and a Frobenius-norm error. We inspect the keys and confirm the
reported error matches our own comparison to `U_exact`.

In [3]:
with quiet():
    res = ua.TrotterAlgorithm().run(H, t, 0.01, order=1, steps=200, backend="numpy")
print("keys:", list(res.keys()))
U_trot = res["Approximate evolution matrix"]
print("Frobenius error (reported):", format(res["Frobenius norm of error"], ".3e"))
print("Frobenius error (recomputed):",
      format(float(np.linalg.norm(U_trot - U_exact)), ".3e"))
# The algorithm's own exact matrix should agree with our reference expm.
assert np.allclose(res["Exact evolution matrix"], U_exact, atol=1e-8)
print("Verified: algorithm's exact propagator matches our reference expm.")

keys: ['status', 'circuit_path', 'plot', 'circuit', 'Approximate evolution matrix', 'Exact evolution matrix', 'Frobenius norm of error']
Frobenius error (reported): 5.432e-03
Frobenius error (recomputed): 5.432e-03
Verified: algorithm's exact propagator matches our reference expm.


## 3. Error vs. number of steps (order 1 vs. order 2)

At fixed evolution time $t$, first-order Trotter error scales like $\mathcal{O}(1/r)$
and second-order like $\mathcal{O}(1/r^2)$. We sweep the step count $r$ and record the
Frobenius distance $\lVert U_{\text{Trotter}} - U_{\text{exact}}\rVert_F$ for each order.

**Tolerance / claim:** at a fixed step count the second-order error is strictly
smaller than the first-order error.

In [4]:
steps_list = [5, 10, 20, 40, 80]
err1, err2 = [], []
for r in steps_list:
    with quiet():
        r1 = ua.TrotterAlgorithm().run(H, t, 0.01, order=1, steps=r, backend="numpy")
        r2 = ua.TrotterAlgorithm().run(H, t, 0.01, order=2, steps=r, backend="numpy")
    err1.append(float(r1["Frobenius norm of error"]))
    err2.append(float(r2["Frobenius norm of error"]))

print(" steps   order-1 err   order-2 err")
for r, e1, e2 in zip(steps_list, err1, err2):
    print(f" {r:4d}    {e1:.3e}    {e2:.3e}")

# Order-2 must beat order-1 at every fixed step count.
assert all(e2 < e1 for e1, e2 in zip(err1, err2))
# Slopes on a log-log plot should be about -1 (order 1) and -2 (order 2).
p1 = np.polyfit(np.log(steps_list), np.log(err1), 1)[0]
p2 = np.polyfit(np.log(steps_list), np.log(err2), 1)[0]
print(f"\nfitted slopes: order-1 ~ {p1:.2f}  (expect ~ -1),  order-2 ~ {p2:.2f}  (expect ~ -2)")
assert p1 < -0.8 and p2 < -1.7
print("Verified: order-2 error < order-1 error at every step count.")

 steps   order-1 err   order-2 err
    5    2.190e-01    2.014e-02
   10    1.089e-01    5.005e-03
   20    5.435e-02    1.249e-03
   40    2.716e-02    3.122e-04
   80    1.358e-02    7.805e-05

fitted slopes: order-1 ~ -1.00  (expect ~ -1),  order-2 ~ -2.00  (expect ~ -2)
Verified: order-2 error < order-1 error at every step count.


In [5]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(steps_list, err1, "o-", label="order 1 (Lie)")
ax.loglog(steps_list, err2, "s-", label="order 2 (Strang)")
ax.set_xlabel("Trotter steps  r")
ax.set_ylabel(r"$\|U_{\mathrm{Trotter}}-U_{\mathrm{exact}}\|_F$")
ax.set_title("TFIM (2 qubits), t = 1: Trotter error vs. steps")
ax.legend(); ax.grid(True, which="both", ls=":")
plt.tight_layout(); plt.show()
print("order-1 slope ~ -1, order-2 slope ~ -2 (as annotated above)")

order-1 slope ~ -1, order-2 slope ~ -2 (as annotated above)


## 4. Randomized compilation: qDrift

`ua.QDriftAlgorithm` samples terms of $H$ randomly (weighted by their coefficients)
so the gate count is independent of the number of terms. Its error decreases like
$\mathcal{O}(1/\sqrt{N})$ in the number of samples $N$ — slower per-sample than
Trotter but cheap when $H$ has many terms.

**Tolerance / claim:** with enough samples qDrift reaches error below $0.15$.

In [6]:
with quiet():
    rq = ua.QDriftAlgorithm().run(H, t, 0.05, steps=8000, backend="numpy")
err_qdrift = float(rq["Frobenius norm of error"])
print("qDrift Frobenius error (8000 samples):", format(err_qdrift, ".3e"))
assert err_qdrift < 0.15
print("Verified: qDrift propagator is within 0.15 (Frobenius) of the exact evolution.")

qDrift Frobenius error (8000 samples): 2.010e-02
Verified: qDrift propagator is within 0.15 (Frobenius) of the exact evolution.


## 5. Your turn
- Extend $H$ to a 3-qubit TFIM chain and confirm the same order-1 vs. order-2
  ordering holds (the matrices become $8\times 8$).
- For fixed target error, read off how many steps each order needs — the
  second-order formula needs far fewer, illustrating why higher-order product
  formulas dominate in practice.
- Compare the qDrift sample count needed to match a 40-step order-2 Trotter error;
  this is the crossover where randomized compilation stops being competitive for
  few-term Hamiltonians.

Lecture 12 replaces product formulas with **block-encoding** methods (Taylor-series
LCU and QSP), whose cost scales far better with the target precision $\epsilon$.